# colab40 — master evaluation for Chapter 4

Produces **every number and every figure Chapter 4 needs**, from one run, on one set of evaluation
objects. Design decisions are recorded in `CH4_CH5_RUN_DESIGN_2026-08-25.md`.

**Why one run.** SNNEED, ESM-2 and Dice currently come from the run of record while the length
baseline comes from the length-constraint run, whose 3Di MAP@10 is 0.508 against 0.515. A shared
chapter-4 table across two runs is not honest. This notebook replaces both.

### Stages
* **A — artefacts.** Collections, exact relevance sets, decile-balanced pair sets, environment
  capture. Persisted to Drive, so stages B and C and the later architecture notebook never rebuild
  them. Every number Chapter 3 claims is asserted here.
* **B — methods.** SNNEED over seeds 0, 1, 2; ESM-2, Dice and Length once each.
* **C — outputs.** Heatmaps, score-vs-truth panels, chance floor, and one results table.

⚠ **Runtime is dominated by Stage A** — one full pairwise scan per collection, SS being the slow
one. It runs **once**; afterwards the cached artefacts load from Drive in seconds.

⚠ **Do not "improve" the padding.** Sequences are padded to a fixed width of 200 and pooling runs
over the padded width. That is the deployed behaviour and the length-constraint run found it
load-bearing. Dynamic per-batch padding is a *different model*.

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')
!pip install rapidfuzz --quiet

## Constants, palette and cache

Constants are verbatim from the run of record. The palette, the labels and their order are locked
in the design file — **exact capitalisation, exact order, in every figure.**

In [ ]:
import os, json, time, pickle
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

DATA_DIR = 'sampledata/cath'

# --- IDENTICAL to the run of record ---------------------------------------
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
RANGE_LOW, RANGE_HIGH = 0.30, 0.70          # Table 3.1
RESCUED = {'4z0mC02', '3qkaE02'}
N_TRAIN, EPOCHS, LR = 30_000, 30, 1e-3
SEEDS = [0, 1, 2]
STRAT_CAND, STRAT_PER_BIN = 200_000, 400
PAIR_SEED = 999
SYN_PERTURB, SYN_INDEP, SYN_SEED = 20_000, 8_000, 20260810
ESM2_MODEL = 'facebook/esm2_t12_35M_UR50D'
TIE_REPS, TIE_SEED = 200, 1234              # Section 3.6.5

# --- LOCKED labels, order and colours (design file section 1) --------------
METHODS  = ['SNNEED', 'ESM-2', 'Dice', 'Length']
DATASETS = ['Synth', '3Di', 'SS', 'AA']
COLOUR = {'SNNEED': '#C026A6', 'ESM-2': '#2CA02C', 'Dice': '#9E9E9E', 'Length': '#8C564B',
          'Synth': '#FF7F0E', '3Di': '#0072B2', 'SS': '#D62728', 'AA': '#4D4D4D'}

AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s)
is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

# --- Drive cache -----------------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/thesis_artefacts'
except Exception as e:
    CACHE = '/content/thesis_artefacts'
    print('Drive not mounted, caching locally (will NOT survive the session):', e)
os.makedirs(CACHE, exist_ok=True)
print('artefact cache:', CACHE)

## Stage A1 — collections

Same filter as the run of record. **If an assert fails, stop**: the rest of the notebook would
describe a different dataset than Chapter 3.

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))

COLL = {
    'AA':  [s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)],
    'SS':  [s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)],
    '3Di': [s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)],
}
assert len(COLL['AA'])  == 10_501, 'AA collection differs from the run of record - STOP'
assert len(COLL['SS'])  == 10_497, 'SS collection differs from the run of record - STOP'
assert len(COLL['3Di']) == 10_501, '3Di collection differs from the run of record - STOP'
for f in ['AA', 'SS', '3Di']:
    print(f'  {f:<4} {len(COLL[f]):>6,} sequences')
print('matches Section 3.4')

In [ ]:
def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN + 1)); return ''.join(rng.choice(list(abc), size=L))

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0:            op = 'ins'
        elif len(s) >= MAX_LEN:    op = rng.choice(['sub', 'del'])
        else:                      op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def build_synth_eval(seed=SYN_SEED):
    '''Synthetic EVALUATION set: 20,000 altered + 8,000 independent, decile-balanced.
       Note it differs from training in how the edit count is drawn - integer here.'''
    r = np.random.default_rng(seed); recs = []
    for _ in range(SYN_PERTURB):
        base = rand_seq(AA_ALPHABET, r)
        part = perturb(base, int(r.integers(0, len(base) + 1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN: recs.append((base, part))
    for _ in range(SYN_INDEP):
        recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]
    nl_all = np.array([x[2] for x in recs])
    bins = np.clip(np.digitize(nl_all, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:STRAT_PER_BIN].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, l = recs[int(idx)]
        I.append(len(seqs)); seqs.append(a)
        J.append(len(seqs)); seqs.append(b)
        NL.append(l)
    return seqs, np.array(I), np.array(J), np.array(NL), nl_all

print('generating the Synth evaluation collection - about a minute...')
COLL['Synth'], SYN_I, SYN_J, SYN_NL, SYN_NL_ALL = build_synth_eval()
print(f'  {len(COLL["Synth"]):,} sequences, {len(SYN_NL):,} balanced pairs '
      f'(run of record: 7,296 and 3,648)')
assert len(COLL['Synth']) == 7_296, 'Synth Eval collection differs from the run of record - STOP'
assert len(SYN_NL) == 3_648, 'Synth balanced pair count differs from the run of record - STOP'
print('matches Section 3.4')

## Stage A2 — exact relevance sets

⚠ **The expensive cell.** One full pairwise pass per collection. Cached to Drive: delete
`relevance_sets.pkl` there to force a rebuild.

Asserts the counts Chapter 3 states: high-similarity pairs **AA 5 · 3Di 6,009 · SS 623,077**, and
eligible queries **Synth 2,410 · 3Di 347 · SS 10,002 · AA 10**.

In [ ]:
REL_PATH = f'{CACHE}/relevance_sets.pkl'

def build_relevance(seqs, block=1024, tag=''):
    N = len(seqs); lens = np.array([len(s) for s in seqs])
    T_high, pos = {}, []
    t0 = time.time()
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        D = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - D / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= RANGE_HIGH)[0]
            if hi.size:
                T_high[i] = hi.astype(np.int32)
                for j in hi:
                    if j > i: pos.append((i, int(j), float(row[j])))
        print(f'    {tag} {r1:>6,}/{N:,}  ({time.time()-t0:.0f}s)', end='\r')
    print()
    return dict(T_high=T_high, pos_pairs=pos)

if os.path.exists(REL_PATH):
    with open(REL_PATH, 'rb') as fh: REL = pickle.load(fh)
    print('loaded cached relevance sets from Drive')
else:
    REL = {}
    for f in DATASETS:
        print(f'{f}: exact relevance-set scan over {len(COLL[f]):,} sequences')
        REL[f] = build_relevance(COLL[f], tag=f)
    with open(REL_PATH, 'wb') as fh: pickle.dump(REL, fh)
    print('cached to Drive')

for f in DATASETS:
    print(f'  {f:<6} high-similarity pairs = {len(REL[f]["pos_pairs"]):>8,}   '
          f'eligible queries = {len(REL[f]["T_high"]):>7,}')

assert len(REL['AA']['pos_pairs'])  == 5,       'AA high-similarity pair count differs - STOP'
assert len(REL['3Di']['pos_pairs']) == 6_009,   '3Di high-similarity pair count differs - STOP'
assert len(REL['SS']['pos_pairs'])  == 623_077, 'SS high-similarity pair count differs - STOP'
for f, n in [('Synth', 2_410), ('3Di', 347), ('SS', 10_002), ('AA', 10)]:
    assert len(REL[f]['T_high']) == n, f'{f} eligible-query count differs from Table 3.8 - STOP'
print('\nmatches Section 3.6 and Table 3.8')

## Stage A3 — decile-balanced pair sets

200,000 candidate pairs per CATH collection, self-pairs discarded, all high-similarity pairs
injected, ten equal-width intervals, at most 400 retained per interval. Synth is already built
pairwise above.

⚠ **This settles two open questions in Chapter 3**: 3Di's evaluation-set size (3,668 derived
against 3,692 reported) and the high-range column of Table 3.8, currently 1,200 by construction.
Whatever comes out here is the truth; **the prose follows the run, not the other way round.**

In [ ]:
STRAT_PATH = f'{CACHE}/balanced_pairs.pkl'

def build_balanced(feed, rng):
    seqs = COLL[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if REL[feed]['pos_pairs']:
        pa = np.array(REL[feed]['pos_pairs'], dtype=float)
        a  = np.concatenate([a, pa[:, 0].astype(np.int64)])
        b  = np.concatenate([b, pa[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, pa[:, 2]])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9)
    take, supply = [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]; supply.append(int(idx.size))
        if idx.size: take.extend(rng.permutation(idx)[:STRAT_PER_BIN].tolist())
    take = np.array(take, dtype=np.int64)
    return dict(i=a[take], j=b[take], nl=nl[take], supply=supply)

if os.path.exists(STRAT_PATH):
    with open(STRAT_PATH, 'rb') as fh: STRAT = pickle.load(fh)
    print('loaded cached balanced pair sets from Drive')
else:
    rng = np.random.default_rng(PAIR_SEED)
    STRAT = {f: build_balanced(f, rng) for f in ['AA', 'SS', '3Di']}
    with open(STRAT_PATH, 'wb') as fh: pickle.dump(STRAT, fh)
    print('cached to Drive')

STRAT['Synth'] = dict(i=SYN_I, j=SYN_J, nl=SYN_NL, supply=None)

print(f'\n{"dataset":<8}{"pairs":>8}{"high-range":>12}   (Table 3.8 records 3,648 / 3,668? / 4,000 / 1,216)')
SIZES = {}
for f in DATASETS:
    nl = STRAT[f]['nl']; hi = int((nl >= RANGE_HIGH).sum())
    SIZES[f] = dict(pairs=int(len(nl)), high=hi, queries=len(REL[f]['T_high']))
    print(f'  {f:<6}{len(nl):>8,}{hi:>12,}')
print('\n⚠ Compare against Table 3.8. Where they differ, the TABLE is what changes.')

In [ ]:
# --- environment capture: keeps Section 3.7 true --------------------------
import platform, scipy, sklearn, matplotlib, rapidfuzz
try:
    import transformers; TRANSFORMERS_V = transformers.__version__
except Exception:
    TRANSFORMERS_V = None
ENV = dict(captured_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
           python=platform.python_version(), platform=platform.platform(),
           torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__,
           scipy=scipy.__version__, sklearn=sklearn.__version__,
           rapidfuzz=rapidfuzz.__version__, matplotlib=matplotlib.__version__,
           transformers=TRANSFORMERS_V,
           cuda_available=torch.cuda.is_available(),
           cuda_device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           cuda_version=torch.version.cuda)
print(json.dumps(ENV, indent=2))
print('\n⚠ Section 3.7 Table 3.9 must match these. transformers was MISSING from the old record;')
print('  if it is not None above, add it as a row.')

## Stage B — metrics

Definitions follow Section 3.6 exactly.

* Spearman with average ranks, overall and per range. ⚠ **A range with fewer than 10 pairs is not
  estimated** — AA's high range holds 5, and its cell stays blank (design decision 2026-08-25).
* AUROC with the high range as the positive class.
* MAP@10. For SNNEED and ESM-2 the scores are continuous, so a plain top-10 is well defined. For
  **Dice and Length the scores tie in large groups**, so the reported value is the mean over 200
  random orderings within tied groups, exactly as Section 3.6.5 states.
* RMSE over the high range, **SNNEED only** — the only method whose output is on the $s_{Lev}$ scale.

In [ ]:
RANGES = {'far':  lambda nl: nl < RANGE_LOW,
          'mid':  lambda nl: (nl >= RANGE_LOW) & (nl < RANGE_HIGH),
          'high': lambda nl: nl >= RANGE_HIGH}
MIN_N_RANGE = 10

def rho(sim, nl):
    if len(nl) < MIN_N_RANGE or np.ptp(nl) == 0: return np.nan
    r = spearmanr(sim, nl).correlation
    return float(r) if r == r else np.nan

def auroc(sim, nl):
    y = (nl >= RANGE_HIGH).astype(int)
    return float(roc_auc_score(y, sim)) if 0 < y.sum() < len(y) else np.nan

def rmse_high(pred, nl):
    m = nl >= RANGE_HIGH
    return float(np.sqrt(np.mean((pred[m] - nl[m]) ** 2))) if m.sum() else np.nan

def _ap(rel_seq, R, k=10):
    c = np.cumsum(rel_seq); prec = c / np.arange(1, k + 1)
    return float((rel_seq * prec).sum() / min(R, k))

def map10_untied(score_rows, T_high, k=10):
    '''score_rows(query_index) -> full score vector. For continuous scores only.'''
    aps = []
    for qi, rel in T_high.items():
        s = score_rows(qi); s[qi] = -np.inf
        top = np.argpartition(-s, k)[:k]; top = top[np.argsort(-s[top])]
        ts = set(rel.tolist())
        aps.append(_ap(np.array([1 if o in ts else 0 for o in top]), len(rel), k))
    return float(np.mean(aps)) if aps else np.nan

def map10_tied(score_rows, T_high, k=10, reps=TIE_REPS, seed=TIE_SEED):
    '''Mean over `reps` random orderings within tied groups (Section 3.6.5).
       Only the tied group straddling rank k is resampled; everything above it is fixed.'''
    rng = np.random.default_rng(seed); prepped = []
    for qi, rel in T_high.items():
        s = score_rows(qi).astype(np.float64); s[qi] = -np.inf
        mask = np.zeros(len(s), dtype=bool); mask[rel] = True
        kth = -np.partition(-s, k - 1)[k - 1]
        above = np.where(s > kth)[0]; tied = np.where(s == kth)[0]
        prepped.append((s[above].copy(), mask[above].copy(),
                        int(len(tied)), int(mask[tied].sum()), len(rel)))
    out = np.empty(reps)
    for r in range(reps):
        aps = np.empty(len(prepped))
        for t, (asc, arel, n_tie, r_tie, R) in enumerate(prepped):
            na = len(asc)
            if na:
                order = np.lexsort((rng.random(na), -asc)); head = arel[order].astype(np.int8)
            else:
                head = np.zeros(0, dtype=np.int8)
            m = k - na; tail = np.zeros(0, dtype=np.int8)
            if m > 0 and n_tie > 0:
                take = min(m, n_tie)
                h = rng.hypergeometric(r_tie, n_tie - r_tie, take)
                tail = np.zeros(take, dtype=np.int8)
                if h: tail[rng.choice(take, size=h, replace=False)] = 1
            seq = np.concatenate([head, tail])
            if len(seq) < k: seq = np.concatenate([seq, np.zeros(k - len(seq), dtype=np.int8)])
            aps[t] = _ap(seq[:k], R, k)
        out[r] = aps.mean()
    return float(out.mean()), float(out.std(ddof=1))

def record(method, dataset, sim, nl, m10, m10_sd=np.nan, pred=None):
    rec = dict(method=method, dataset=dataset, spearman=rho(sim, nl),
               auroc=auroc(sim, nl), map10=m10, map10_tie_sd=m10_sd,
               rmse_high=rmse_high(pred, nl) if pred is not None else np.nan,
               n_pairs=int(len(nl)), n_queries=len(REL[dataset]['T_high']))
    for name, sel in RANGES.items():
        m = sel(nl)
        rec[f'spearman_{name}'] = rho(sim[m], nl[m]) if m.sum() >= MIN_N_RANGE else np.nan
        rec[f'n_{name}'] = int(m.sum())
    return rec

print('metrics defined')

## Stage B1 — SNNEED, three seeds

Architecture and training verbatim from the run of record: fixed padding to 200, pooling over the
padded width, chord readout, unweighted MSE, Adam at $10^{-3}$, batch 128, 30 epochs.

In [ ]:
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX] * (MAX_LEN - len(idx))
    return torch.tensor(idx, dtype=torch.long)

class EncPool(nn.Module):
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64 * K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)

class RegModel(nn.Module):
    '''Deployed SNNEED: chord readout, no head. The readout has no parameters.'''
    def __init__(s, enc): super().__init__(); s.encoder = enc
    def forward(s, a, b):
        ea, eb = s.encoder(a), s.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0

class DS(Dataset):
    def __init__(s, pp): s.p = pp
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), torch.tensor(l, dtype=torch.float32)

def build_train_pairs(n, seed):
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd)
        t = float(rng.uniform(0, 1)); k = max(0, int(round((1 - t) * L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    return pairs

def train_snneed(seed):
    pairs = build_train_pairs(N_TRAIN, seed)
    torch.manual_seed(seed)
    model = RegModel(EncPool()).to(device)
    dl = DataLoader(DS(pairs), batch_size=BS, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), LR); model.train(); t0 = time.time()
    for ep in range(EPOCHS):
        tot = 0.0
        for xa, xb, y in dl:
            xa, xb, y = xa.to(device), xb.to(device), y.to(device)
            loss = F.mse_loss(model(xa, xb), y)          # UNWEIGHTED - Section 3.3.2
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(y)
        if (ep + 1) % 10 == 0:
            print(f'    seed {seed} epoch {ep+1:>2}/{EPOCHS} loss {tot/len(pairs):.5f} '
                  f'({time.time()-t0:.0f}s)')
    return model.eval()

@torch.no_grad()
def embed(model, feed, bs=256):
    out = []
    for i in range(0, len(COLL[feed]), bs):
        x = torch.stack([encode_pad(s) for s in COLL[feed][i:i+bs]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

ROWS = []
for seed in SEEDS:
    print(f'SNNEED seed {seed}')
    model = train_snneed(seed)
    for f in DATASETS:
        E = embed(model, f); P = STRAT[f]
        sim = np.sum(E[P['i']] * E[P['j']], axis=1)     # cosine == chord readout, monotone
        pred = 1.0 - np.linalg.norm(E[P['i']] - E[P['j']], axis=1) / 2.0
        Et = torch.as_tensor(E, device=device)
        rows = lambda qi: (Et[qi] @ Et.t()).cpu().numpy().astype(np.float64)
        r = record('SNNEED', f, sim, P['nl'], map10_untied(rows, REL[f]['T_high']), pred=pred)
        r['seed'] = seed; ROWS.append(r)
        print(f'    {f:<6} rho={r["spearman"]:+.3f} AUROC={r["auroc"]:.3f} '
              f'MAP@10={r["map10"]:.3f} RMSE_high={r["rmse_high"]:.3f}')

## Stage B2 — ESM-2, Dice and Length

In [ ]:
@torch.no_grad()
def esm_embed(feed, bs=32):
    from transformers import AutoTokenizer, AutoModel
    if 'mdl' not in globals():
        globals()['tok'] = AutoTokenizer.from_pretrained(ESM2_MODEL)
        globals()['mdl'] = AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
    seqs = COLL[feed]; order = np.argsort([len(s) for s in seqs]); out = [None] * len(seqs)
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]; batch = [seqs[j] for j in idx]
        enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=True).to(device)
        h = mdl(**enc).last_hidden_state
        mask = enc['attention_mask'].clone(); mask[:, 0] = 0          # drop BOS
        for r, l in enumerate(enc['attention_mask'].sum(1)): mask[r, l-1] = 0   # drop EOS
        m = mask.unsqueeze(-1).float()
        e = F.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=1).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    return np.stack(out).astype(np.float32)

for f in DATASETS:
    E = esm_embed(f); P = STRAT[f]
    sim = np.sum(E[P['i']] * E[P['j']], axis=1)
    Et = torch.as_tensor(E, device=device)
    rows = lambda qi: (Et[qi] @ Et.t()).cpu().numpy().astype(np.float64)
    r = record('ESM-2', f, sim, P['nl'], map10_untied(rows, REL[f]['T_high']))
    ROWS.append(r)
    print(f'  ESM-2  {f:<6} rho={r["spearman"]:+.3f} AUROC={r["auroc"]:.3f} MAP@10={r["map10"]:.3f}')

In [ ]:
def gram_matrix(seqs, k=3):
    vocab, rows, cols = {}, [], []
    for i, s in enumerate(seqs):
        for g in {s[t:t+k] for t in range(len(s) - k + 1)}:
            rows.append(i); cols.append(vocab.setdefault(g, len(vocab)))
    M = sparse.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                          shape=(len(seqs), max(1, len(vocab))))
    return M, np.asarray(M.sum(1)).ravel()

for f in DATASETS:
    M, sz = gram_matrix(COLL[f]); P = STRAT[f]
    inter = np.asarray(M[P['i']].multiply(M[P['j']]).sum(1)).ravel().astype(float)
    sim = 2 * inter / np.maximum(sz[P['i']] + sz[P['j']], 1e-9)
    def rows(qi, M=M, sz=sz):
        it = np.asarray(M.dot(M[qi].T).todense()).ravel().astype(np.float64)
        return 2 * it / np.maximum(sz + sz[qi], 1e-9)
    m, sd = map10_tied(rows, REL[f]['T_high'])          # Dice ties - Section 3.6.5
    ROWS.append(record('Dice', f, sim, P['nl'], m, sd))
    print(f'  Dice   {f:<6} MAP@10={m:.3f} (tie sd {sd:.5f})')

for f in DATASETS:
    lens = np.array([len(s) for s in COLL[f]], dtype=np.float64); P = STRAT[f]
    sim = (np.minimum(lens[P['i']], lens[P['j']]) / np.maximum(lens[P['i']], lens[P['j']]))
    def rows(qi, lens=lens):
        return np.minimum(lens, lens[qi]) / np.maximum(lens, lens[qi])
    m, sd = map10_tied(rows, REL[f]['T_high'])          # Length ties - Section 3.6.5
    ROWS.append(record('Length', f, sim, P['nl'], m, sd))
    print(f'  Length {f:<6} MAP@10={m:.3f} (tie sd {sd:.5f})')

RES = pd.DataFrame(ROWS)
AGG = (RES.groupby(['method', 'dataset'], as_index=False)
          .agg({c: 'mean' for c in RES.columns if RES[c].dtype.kind == 'f'}))
SD  = (RES[RES.method == 'SNNEED'].groupby('dataset', as_index=False)
          .agg(spearman_sd=('spearman', 'std'), auroc_sd=('auroc', 'std'),
               map10_sd=('map10', 'std'), rmse_sd=('rmse_high', 'std')))
RES.to_csv('colab40_results_raw.csv', index=False)
AGG.to_csv('colab40_results_mean.csv', index=False)
print(SD)
AGG

## Stage C — figures

Grading is locked in the design file:

| Metric | Scale |
|---|---|
| Spearman | blue $-1$ → white $0$ → red $+1$ |
| MAP@10, AUROC | white $0$ → red $1$ |
| RMSE | **reversed** — low is red, so red always means good |

Every cell is annotated to **2 decimal places**.

In [ ]:
def heat(ax, values, title, kind, annot_extra=None, cbar=True, cbar_label=None):
    '''values: dict[(method, dataset)] -> float. kind in {spearman, unit, rmse}.'''
    Mx = np.full((len(METHODS), len(DATASETS)), np.nan)
    for r, m in enumerate(METHODS):
        for c, d in enumerate(DATASETS):
            v = values.get((m, d), np.nan)
            Mx[r, c] = v if v is not None else np.nan
    if kind == 'spearman':
        im = ax.imshow(Mx, cmap='bwr', norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))
    elif kind == 'rmse':
        im = ax.imshow(Mx, cmap='Reds_r', vmin=0, vmax=np.nanmax(Mx) if np.isfinite(Mx).any() else 1)
    else:
        im = ax.imshow(Mx, cmap='Reds', vmin=0, vmax=1)
    ax.set_xticks(range(len(DATASETS)), DATASETS)
    ax.set_yticks(range(len(METHODS)), METHODS)
    ax.set_title(title, fontsize=10)
    for r in range(len(METHODS)):
        for c in range(len(DATASETS)):
            v = Mx[r, c]
            if not np.isfinite(v):
                ax.text(c, r, '--', ha='center', va='center', fontsize=8, color='0.55')
                continue
            shade = im.cmap(im.norm(v))[:3]
            col = 'white' if (0.299*shade[0] + 0.587*shade[1] + 0.114*shade[2]) < 0.55 else 'black'
            txt = f'{v:.2f}'
            if annot_extra and (METHODS[r], DATASETS[c]) in annot_extra:
                txt += annot_extra[(METHODS[r], DATASETS[c])]
            ax.text(c, r, txt, ha='center', va='center', fontsize=9, color=col)
    if cbar:
        cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.ax.tick_params(labelsize=8)
        if cbar_label: cb.set_label(cbar_label, fontsize=8)
        if kind == 'spearman': cb.set_ticks([-1, -0.5, 0, 0.5, 1])
        elif kind == 'unit':   cb.set_ticks([0, 0.25, 0.5, 0.75, 1])
    return im

def vals(col):
    return {(m, d): float(AGG[(AGG.method == m) & (AGG.dataset == d)][col].iloc[0])
            for m in METHODS for d in DATASETS}

fig, ax = plt.subplots(figsize=(5.6, 3.4))
heat(ax, vals('spearman'), 'Spearman correlation', 'spearman', cbar_label='$\\rho$')
fig.tight_layout(); fig.savefig('colab40_spearman.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.4))
for ax, rname in zip(axes, ['far', 'mid', 'high']):
    heat(ax, vals(f'spearman_{rname}'), f'{rname} range', 'spearman')
    if rname != 'far': ax.set_yticks([])
fig.suptitle('Spearman correlation by similarity range', fontsize=11)
fig.tight_layout()
fig.savefig('colab40_spearman_by_range.png', dpi=200, bbox_inches='tight'); plt.show()
print('⚠ AA/high is blank by design: it holds 5 pairs and is not estimated. Say so in the caption.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.4))
heat(axes[0], vals('map10'), 'MAP@10', 'unit', cbar_label='MAP@10')
heat(axes[1], vals('auroc'), 'AUROC', 'unit', cbar_label='AUROC'); axes[1].set_yticks([])
fig.tight_layout(); fig.savefig('colab40_map_auroc.png', dpi=200, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(5.6, 1.8))
rm = {('SNNEED', d): vals('rmse_high')[('SNNEED', d)] for d in DATASETS}
Mx = np.array([[rm[('SNNEED', d)] for d in DATASETS]])
im = ax.imshow(Mx, cmap='Reds_r', vmin=0, vmax=np.nanmax(Mx))
ax.set_xticks(range(len(DATASETS)), DATASETS); ax.set_yticks([0], ['SNNEED'])
for c, d in enumerate(DATASETS):
    ax.text(c, 0, f'{Mx[0, c]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('RMSE, high range (lower is better - scale reversed)', fontsize=9)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.ax.tick_params(labelsize=8); cb.set_label('RMSE', fontsize=8)
fig.tight_layout(); fig.savefig('colab40_rmse.png', dpi=200, bbox_inches='tight'); plt.show()

## Score against truth — SNNEED and ESM-2 only

Both score by cosine, so a shared axis is legitimate. Dice and Length are **not** included: their
scores are not cosines and the scales are incommensurable, so a shared panel would imply a
calibration that does not exist.

Design reused from the earlier pairwise-scatter figure: 1,200 pairs sampled per dataset, coloured
in the locked palette.

In [ ]:
# Score against truth. Cosine readout for BOTH methods - the same quantity the Spearman, AUROC and
# MAP@10 numbers are computed from, so the figure shows exactly what the table reports.
# Scatter in the background, the binned mean of that scatter drawn over it, one line per dataset.
# The binned mean is taken over ALL pairs, not over the plotted subsample.
SCATTER_N, SCATTER_SEED = 1200, 17
BIN_EDGES, MIN_PER_BIN = np.linspace(0, 1, 21), 10

last = {}
last['SNNEED'] = {f: embed(model, f) for f in DATASETS}          # final seed's encoder
last['ESM-2']  = {f: esm_embed(f) for f in DATASETS}

def cosine_of(E, P):
    return np.sum(E[P['i']] * E[P['j']], axis=1)

def binned_mean(x, y, edges=BIN_EDGES, min_n=MIN_PER_BIN):
    idx = np.clip(np.digitize(x, edges) - 1, 0, len(edges) - 2)
    cx, cy = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_n:
            cx.append(0.5 * (edges[b] + edges[b + 1])); cy.append(float(y[m].mean()))
    return np.array(cx), np.array(cy)

def scatter_panel(ax, method, feeds, show_legend=False):
    for f in feeds:
        P = STRAT[f]; s = cosine_of(last[method][f], P); nl = P['nl']
        r = np.random.default_rng(SCATTER_SEED + sum(ord(c) for c in f))
        idx = np.arange(len(nl))
        if len(idx) > SCATTER_N: idx = r.choice(idx, SCATTER_N, replace=False)
        ax.scatter(nl[idx], s[idx], s=6, alpha=0.13, linewidths=0, color=COLOUR[f])
        bx, by = binned_mean(nl, s)
        ax.plot(bx, by, '-', lw=4.0, color='white', alpha=0.7, zorder=2)   # halo
        ax.plot(bx, by, '-', lw=2.2, color=COLOUR[f], label=f, zorder=3)
    ax.set_xlabel('exact normalised Levenshtein similarity', fontsize=9)
    ax.set_ylabel('cosine similarity', fontsize=9)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    if show_legend: ax.legend(frameon=False, fontsize=8)

# --- overview: both methods, all datasets overlaid. Independent y-axes, both fully labelled. ---
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.4))
for ax, m in zip(axes, ['SNNEED', 'ESM-2']):
    scatter_panel(ax, m, DATASETS, show_legend=(m == 'ESM-2'))
    ax.set_title(m, fontsize=11)
fig.tight_layout(); fig.savefig('colab40_score_vs_truth.png', dpi=200, bbox_inches='tight'); plt.show()

# --- per dataset: 2 rows (method) x 4 columns (dataset). Every panel keeps its own scale. ---
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for row, m in enumerate(['SNNEED', 'ESM-2']):
    for col, f in enumerate(DATASETS):
        ax = axes[row, col]
        scatter_panel(ax, m, [f])
        P = STRAT[f]; s = cosine_of(last[m][f], P)
        rho = spearmanr(s, P['nl']).correlation
        ax.set_title(f'{m}  x  {f}   ($\\rho$ = {rho:+.3f})', fontsize=10)
fig.tight_layout(); fig.savefig('colab40_score_vs_truth_grid.png', dpi=200, bbox_inches='tight')
plt.show()

# How flat is each curve in the high range against the mid range? A collapse is a small ratio.
print('slope of the binned mean, mid range vs high range:')
for m in ['SNNEED', 'ESM-2']:
    for f in DATASETS:
        P = STRAT[f]; s = cosine_of(last[m][f], P)
        bx, by = binned_mean(P['nl'], s)
        mid = (bx >= RANGE_LOW) & (bx < RANGE_HIGH); hi = bx >= RANGE_HIGH
        if mid.sum() > 1 and hi.sum() > 1:
            sm = np.polyfit(bx[mid], by[mid], 1)[0]; sh = np.polyfit(bx[hi], by[hi], 1)[0]
            print(f'  {m:<7}{f:<6} mid {sm:+.3f}   high {sh:+.3f}   ratio {sh / sm:+.2f}')
        else:
            print(f'  {m:<7}{f:<6} too few populated bins (AA holds 11 mid and 5 high pairs)')

## Chance floor

**One reference point per alphabet**, so a score can be read as above chance or not. It matters
most for SS: two random three-letter strings are far more similar than two random twenty-letter
strings, so SS's observed median and AA's cannot be read against the same floor.

Two nulls per dataset, both with the length distribution matched to the real collection:

1. **independent** — random strings over that dataset's own alphabet;
2. **shuffled** — the real sequences with their symbols permuted, which preserves length and
   composition and destroys order.

⚠ These numbers are persisted. The previous floor measurement was lost because it was printed and
never saved.

In [ ]:
FLOOR_N, FLOOR_SEED = 20_000, 4242
ALPHABET = {'Synth': AA_ALPHABET, 'AA': AA_ALPHABET, '3Di': AA_ALPHABET, 'SS': SS_ALPHABET}

def floors(feed, n=FLOOR_N, seed=FLOOR_SEED):
    r = np.random.default_rng(seed); seqs = COLL[feed]
    lens = np.array([len(s) for s in seqs]); abc = list(ALPHABET[feed])
    La, Lb = r.choice(lens, n), r.choice(lens, n)              # length-matched
    indep = np.array([norm_lev(''.join(r.choice(abc, size=a)), ''.join(r.choice(abc, size=b)))
                      for a, b in zip(La, Lb)])
    ia, ib = r.integers(0, len(seqs), n), r.integers(0, len(seqs), n)
    keep = ia != ib; ia, ib = ia[keep], ib[keep]
    shuf = np.array([norm_lev(''.join(r.permutation(list(seqs[i]))),
                              ''.join(r.permutation(list(seqs[j])))) for i, j in zip(ia, ib)])
    obs = STRAT[feed]['nl']
    return dict(independent_median=float(np.median(indep)),
                independent_p2_5=float(np.percentile(indep, 2.5)),
                independent_p97_5=float(np.percentile(indep, 97.5)),
                shuffled_median=float(np.median(shuf)),
                observed_median_balanced=float(np.median(obs)),
                alphabet_size=len(abc), n=int(n))

FLOOR = {f: floors(f) for f in DATASETS}
FL = pd.DataFrame(FLOOR).T
print(FL.round(3))
print('\n⚠ Read every dataset against ITS OWN floor. A single global floor is meaningless here.')

fig, ax = plt.subplots(figsize=(6.4, 3.4))
x = np.arange(len(DATASETS))
ax.bar(x - 0.2, [FLOOR[f]['independent_median'] for f in DATASETS], 0.38,
       color=[COLOUR[f] for f in DATASETS], alpha=0.45, label='independent strings')
ax.bar(x + 0.2, [FLOOR[f]['shuffled_median'] for f in DATASETS], 0.38,
       color=[COLOUR[f] for f in DATASETS], alpha=0.95, label='shuffled sequences')
ax.set_xticks(x, DATASETS); ax.set_ylabel('median normalised Levenshtein similarity')
ax.set_title('Chance floor per alphabet', fontsize=10)
ax.legend(frameon=False, fontsize=9); ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); fig.savefig('colab40_chance_floor.png', dpi=200, bbox_inches='tight'); plt.show()

## Save everything

In [ ]:
summary = dict(
    config=dict(seeds=SEEDS, epochs=EPOCHS, n_train=N_TRAIN, strat_cand=STRAT_CAND,
                per_bin=STRAT_PER_BIN, pair_seed=PAIR_SEED, tie_reps=TIE_REPS, tie_seed=TIE_SEED),
    environment=ENV,
    collections={f: len(COLL[f]) for f in DATASETS},
    high_similarity_pairs={f: len(REL[f]['pos_pairs']) for f in DATASETS},
    evaluation_sizes=SIZES,
    chance_floor=FLOOR,
    results_mean=AGG.round(6).to_dict('records'),
    results_by_seed=RES.round(6).to_dict('records'))

with open('colab40_master.json', 'w') as fh: json.dump(summary, fh, indent=2, default=float)
json.loads(open('colab40_master.json').read())          # validate
print(json.dumps({k: summary[k] for k in
                  ['collections', 'high_similarity_pairs', 'evaluation_sizes', 'chance_floor']},
                 indent=2, default=float))

FIGS = ['colab40_spearman.png', 'colab40_spearman_by_range.png', 'colab40_map_auroc.png',
        'colab40_rmse.png', 'colab40_score_vs_truth.png', 'colab40_score_vs_truth_grid.png',
        'colab40_chance_floor.png']
OUTS = FIGS + ['colab40_master.json', 'colab40_results_raw.csv', 'colab40_results_mean.csv']

import shutil
for f in OUTS:
    shutil.copy2(f, CACHE); print('saved to Drive:', f)

from google.colab import files
for f in OUTS: files.download(f)

## What to check before anything reaches Chapter 4

1. **The Stage A asserts all passed** — otherwise the run describes different data than Chapter 3.
2. **Table 3.8 against the printed evaluation sizes.** 3Di's total and the high-range column are the
   two known open points. **Where they differ, Chapter 3's table changes, not this run.**
3. **Section 3.7 Table 3.9 against the environment capture**, and add the `transformers` row.
4. **The AA row is an anecdote**: 10 queries, 5 relevant pairs. Never quote an AA retrieval number
   without that count, and never state that AA's high-range Spearman was estimated.
5. **The chance floor is persisted this time** — it is in `colab40_master.json`.